In [ ]:
#PyTorch 版本 以及 Cuda 調用確認
import torch

print(f"PyTorch 版本: {torch.__version__}") # 應該要 >= 2.6.0
print(f"CUDA 是否可用: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"啟動成功！")
    print(f"顯卡名稱: {torch.cuda.get_device_name(0)}")
    print(f"CUDA 驅動上限: {torch.version.cuda}")

PyTorch 版本: 2.11.0+cu126
CUDA 是否可用: True
啟動成功！
顯卡名稱: NVIDIA GeForce RTX 4060 Laptop GPU
CUDA 驅動上限: 12.6


In [ ]:
#文本切片embeding

import json
import chromadb
from chromadb.utils import embedding_functions

# 1. 讀取 JSON 檔案
with open("final_chunks.json", "r", encoding="utf-8") as f:
    final_chunks = json.load(f)

print(f"成功讀取 {len(final_chunks)} 個切片。")

# 2. 設定 Embedding 模型 (BGE-M3)
emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-m3",
    device="cuda"  # 有 NVIDIA 顯卡用 cuda，沒有則改 "cpu"
)

# 3. 初始化本地向量資料庫 (ChromaDB)，當前目錄建立一個名為 hiwin_vector_db 的資料夾
client = chromadb.PersistentClient(path="./hiwin_vector_db")

# 4. 建立或取得 Collection
collection = client.get_or_create_collection(
    name = "hiwin_manual",
    embedding_function = emb_fn,
    metadata = {"hnsw:space": "cosine"}
)

# 5. 執行 Embedding 並存入資料庫
documents = [c['content'] for c in final_chunks]
metadatas = [c['metadata'] for c in final_chunks]
ids = [f"id_{i}" for i in range(len(final_chunks))]

print(f"開始執行 BGE-M3 Embedding 轉換... (共 {len(documents)} 筆)")

# 分批寫入避免記憶體壓力
batch_size = 50
for i in range(0, len(documents), batch_size):
    end = i + batch_size
    collection.add(
        documents = documents[i:end],
        metadatas = metadatas[i:end],
        ids = ids[i:end]
    )
    print(f"已完成: {min(end, len(documents))}/{len(documents)}")

print(f"向量資料庫建置完成！資料夾路徑：./hiwin_vector_db")

成功讀取 87 個切片。


c:\Users\e11338\Desktop\Feed System GAI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<?, ?it/s]


開始執行 BGE-M3 Embedding 轉換... (共 87 筆)
已完成: 50/87
已完成: 87/87
向量資料庫建置完成！資料夾路徑：./hiwin_vector_db


In [ ]:
#Excel 語意欄位建置

import pandas as pd

# 1. 設定檔案路徑
file_path = R"../data/HIWIN_Specs.xlsx"

def process_hiwin_specs(path):
    # 2. 讀取 ALL 工作表
    # 我們在函數內部讀取，這樣 ExcelWriter 才能正確處理檔案鎖定
    df = pd.read_excel(path, sheet_name="ALL")

    # 定義語意合成邏輯
    def create_full_semantic_text(row):
        series = str(row.get('系列', '')).strip()
        model = str(row.get('型號', '')).strip()
        outer_dia = str(row.get('公稱 外徑', '0'))
        lead = str(row.get('導程', '0'))
        ball_dia = str(row.get('珠徑', '0'))
        pcd = str(row.get('PCD', '0'))
        root_dia = str(row.get('根徑', '0'))
        turns = str(row.get('珠卷數', '0'))
        stiffness = str(row.get('剛性 kfg/umk', '0'))
        dyn_load = str(row.get('動負荷 C (kfg)', '0'))
        stat_load = str(row.get('靜負荷 Co (kfg)', '0'))
        
        text = (
            f"上銀 HIWIN 滾珠螺桿型號 {model} (系列: {series})。 "
            f"幾何規格：公稱外徑 {outer_dia}mm，導程 {lead}mm，珠徑 {ball_dia}mm，"
            f"節圓直徑(PCD) {pcd}mm，根徑 {root_dia}mm，珠卷數為 {turns}。 "
            f"機械性能：剛性達 {stiffness} kfg/umk，"
            f"額定動負荷(C)為 {dyn_load} kgf，靜負荷(Co)為 {stat_load} kgf。"
        )
        
        if series == 'FDC':
            text += " 此型號為雙螺帽設計，具備極高剛性，專為重負荷精密機台開發。"
        elif series in ['FSI', 'FSW']:
            text += " 此型號結構輕巧省空間，適合小型自動化設備或精密儀器。"
        elif series == 'FSV':
            text += " 此型號為標準單螺帽設計，傳動效率優異，是工業自動化最通用的選型。"
        elif series == 'RSI':
            text += " 此型號為旋轉螺帽設計，適合長行程且需高速旋轉螺帽的特殊機構。"
            
        return text

    print("正在將全欄位規格轉換為語意描述...")
    df['semantic_text'] = df.apply(create_full_semantic_text, axis=1)

    # 3. 寫回原檔案的 ALL 工作表
    # 使用 mode='a' (append) 與 if_sheet_exists='replace' 來更新特定分頁
    with pd.ExcelWriter(path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        df.to_excel(writer, sheet_name='ALL', index=False)
    
    print(f"處理完成！語意欄位已更新至 {path} 的 ALL 工作表中。")

# 執行函數
process_hiwin_specs(file_path)

正在將全欄位規格轉換為語意描述...
處理完成！語意欄位已更新至 ../data/HIWIN_Specs.xlsx 的 ALL 工作表中。


In [3]:
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions

# 1. 初始化 Embedding 模型
emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-m3",
    device="cuda"
)

# 2. 連接資料庫
client = chromadb.PersistentClient(path="./hiwin_vector_db")

# 3. 建立或取得規格專用的 Collection
spec_collection = client.get_or_create_collection(
    name="hiwin_specs", 
    embedding_function=emb_fn
)

# 4. 讀取 Excel (請確認分頁名稱為 ALL)
path = R"../data/HIWIN_Specs.xlsx"
df = pd.read_excel(path, sheet_name="ALL")
# 重要：處理空值，否則 ChromaDB 的 Metadata 會報錯
df = df.fillna("")
# 5. 準備寫入的資料，直接使用 Excel 裡已經做好的語意欄位
documents = df['semantic_text'].tolist()
# 將除了語意欄位以外的所有原始數據存入 metadata，方便以後過濾或顯示
metadatas = df.drop(columns=['semantic_text']).to_dict('records')
# ID 使用索引 + 型號，確保唯一性
ids = [f"spec_{i}_{row['型號']}" for i, row in df.iterrows()]

# 6. 執行批次存入
print(f"正在寫入 {len(documents)} 筆資料到 hiwin_specs...")
spec_collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f"成功！目前 hiwin_specs 共有 {spec_collection.count()} 筆資料。")

c:\Users\e11338\Desktop\Feed System GAI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 9006.14it/s]


正在寫入 466 筆資料到 hiwin_specs...
成功！目前 hiwin_specs 共有 466 筆資料。
